    ARX
    HARX
    LS (?)
    ARCH-in-mean
+ GARCH, EWMAVariance, EGARCH and TARCH

In [1]:
#VaR tests and vizualization
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import matplotlib.pyplot as plt

from Vares_simulations import *

from Vares import _terminal_returns, historical_var

Vares.ENABLE_TIMING = False

import pickle

#import the USD data 
with open('prices_usd.pkl', 'rb') as f: 
    prices_usd = pickle.load(f)

with open('original_returns_usd.pkl', 'rb') as f: 
    original_returns_usd = pickle.load(f)

with open('log_returns_usd.pkl', 'rb') as f: 
    log_returns_usd = pickle.load(f)    

original_returns_usd_scaled = original_returns_usd * 100
log_returns_usd_scaled = log_returns_usd * 100

from GARCH_VaR_delete_or_merge import fit_GARCH_VaR, normal_GARCH_simulation

import pickle

with open("norm_vars.pkl", "rb") as f:
    _norm_vars = pickle.load(f)

def turn_var_into_real_terms(var_array, prices):
    return [np.exp(log_delta) * price for log_delta, price in zip(var_array, prices)]

In [2]:
from arch.univariate import ARX, HARX, LS, ARCHInMean #mean models 
from arch.univariate import Normal, StudentsT 
# from arch.univariate import Normal, StudentsT, SkewStudent, GeneralizedError - for now modelling these ditribututions is imposible 
from arch.univariate import GARCH, EWMAVariance, EGARCH 

from arch.univariate import ConstantMean

In [31]:
def solve_local_vol_mean_gbm_log(params: GBMParams, 
        T: int,
        n_paths: int = 5000,
        S_0: float = 1,
        seed = None): 
    '''Same as solve_local_vol_gbm(), but handles log returns and allows for dynamic mean process'''

    #the limitation of the presented model is the requirement of the fitted means to be equal to volatilities in amount
    if params.volatility.shape[0] != params.mean.shape[0]:
        raise ValueError('params.volatility.shape[0] has to be equal to params.mean.shape[0]')
        
    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality points. Simulation will assume constant long-term volatility.')

    W_t = stats.norm.rvs(scale=1, size=(n_paths, T), random_state=seed)
    d_log_S = np.zeros(shape=(n_paths, T))
    #infering daily price 
    for t in range(params.volatility.shape[0]): 
        d_log_S[:, t:t+1] = (params.mean[t]
                + params.volatility[t] * W_t[:, t:t+1])
    _ = params.volatility.shape[0] 
    d_log_S[:, _:] = (params.mean[-1]
                + params.volatility[-1] * W_t[:, _:])
    
    cum_log_returns = np.sum(d_log_S, axis=-1)
    return cum_log_returns

def make_random_path_simulator_local_vol_mean_student(
    params: SimParams,
    T: int, 
    nu: float, 
    dist = 't',
    granularity: int = 1000,
):
    '''Simulates returns driven by the Student innovation. Allows for dynamic mean. 
    Returns a batch of simulated data. The simulated data is logged.  Works with the log returns best'''
    if params.volatility.shape[0] != params.mean.shape[0]:
        raise ValueError('params.volatility.shape[0] has to be equal to params.mean.shape[0]')

    if T < params.volatility.shape[0]: 
        raise ValueError('Quantity of forecasted volatility can not exceed number of days to simulte')
    
    elif T > params.volatility.shape[0]:
        print('Number of simulated days exceed number of forecasted volality points. Simulation will assume constant long-term volatility.')

    def simulate(n_paths: int, seed = None):
        if seed is not None:
            rng = np.random.default_rng(seed=seed)
        else:
            rng = np.random.default_rng()

        time_grid = T * granularity
        dt = 1 / granularity
        Z = stats.t.rvs(df=nu, size=(n_paths, time_grid)) #the innovation random variabled
        if nu > 2.0: 
            stoch_comp = Z * np.sqrt(nu-2) / np.sqrt(nu) #scaled to have unit variance 
        else: 
            raise ValueError('Can not model the price process as the innovation variable has non-finite variance')

        #iterating through available volatility points 
        d_log_S = np.zeros(shape=(n_paths, time_grid))
        for t in range(params.volatility.shape[0]):
            _t = t * granularity
            d_log_S[:, _t:_t+granularity] = (
                params.mean[t] * dt + 
                params.volatility[t] * stoch_comp[:, _t:_t+granularity] * np.sqrt(t)
            )
        _T = params.volatility.shape[0] * granularity
        d_log_S[:, _T:] = ((params.mean[-1] * dt) +
                    + params.volatility[-1] * stoch_comp[:, _T:] * np.sqrt(dt))

        cum_log_returns = np.cumsum(d_log_S, axis=-1)
        return cum_log_returns
 
    return simulate


In [32]:
__VOLATILITY__ = {'GARCH': GARCH, 'TARCH': GARCH, 'EWMAVariance': EWMAVariance, 'EGARCH': EGARCH}
__DISTRIBUTIONS__ = {'norm': Normal, 't': StudentsT}
__MEAN__ = {'arx': ARX, 'harx': HARX, 'LS': LS, 'a-i-m': ARCHInMean, 'CM': ConstantMean}

def model_construction(mean_process: str, volatiltiy_process: str, dist_process: str, lags, **kwargs): 
    def simulate(returns, horizon=10, n_paths=50_000, granularity=1, alpha=0.01): 
        vol = __VOLATILITY__[volatiltiy_process](**kwargs)
        if mean_process == 'CM' or mean_process == 'LS':
            model = __MEAN__[mean_process](returns)
        if mean_process == 'harx' or mean_process == 'a-i-m' or mean_process == 'arx':
            model = __MEAN__[mean_process](returns, lags=lags)
        model.volatility = vol 
        model.distribution = __DISTRIBUTIONS__[dist_process]()
        results = model.fit(disp='off')
        h = int(horizon)                          
        forecast = results.forecast(horizon=h, method='simulation')          
        # mu = results.params.get("mu", 0) - used in the case of a single mean 
        mu = forecast.mean.values[0]
        sigma2 = forecast.variance.values[0]
        sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
        parameters = SimParams(volatility=sigma, mean=mu)

        if dist_process == 'norm': 
            sim_returns = solve_local_vol_mean_gbm_log(parameters, 10, n_paths=n_paths)
        
        if dist_process == 't': 
            nu = results.params.get('nu', 0)
            simulator = make_random_path_simulator_local_vol_mean_student(parameters, 10, nu, granularity=granularity)
            sim_returns = _terminal_returns(simulator=simulator, n_paths=n_paths) 

        VaR = historical_var(sim_returns, alpha)
        
        return VaR

    return simulate

    CHECKS

In [5]:
attempt_simulation = model_construction('CM', 'GARCH', 'norm', p=1, q=1, lags=None)

In [6]:
test1 = fit_GARCH_VaR(log_returns_usd_scaled, attempt_simulation)
test2 = fit_GARCH_VaR(log_returns_usd_scaled, normal_GARCH_simulation)

In [7]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(y=test1, mode='lines', name='Constructed generator', line=dict(color='blue')))  
fig.add_trace(go.Scatter(y=test2, mode='lines', name='Original generator', line=dict(color='red'))) 

fig.update_layout(template='plotly_white', showlegend=True)
fig.show()

    TESTS

In [8]:
GARCH_ARX_simulation = model_construction('arx', 'GARCH', 'norm', p=1, q=1, lags=[1, 5, 22])
GARCH_HARX_simulation = model_construction('harx', 'GARCH', 'norm', p=1, q=1, lags=[1, 5, 22])
GARCH_ARX_3_simulation = model_construction('arx', 'GARCH', 'norm', p=1, q=1, lags=3)
GARCH_LS_simulation = model_construction('LS', 'GARCH', 'norm', p=1, q=1, lags=None)
GARCH_AIM_simulation = model_construction('a-i-m', 'GARCH', 'norm', p=1, q=1, lags=None)

In [9]:
__MEAN__['arx'](y=log_returns_usd_scaled, lags=[1,2])

AR(constant: yes, lags: 1, 2, no. of exog: 0, volatility: Constant Variance, distribution: Normal distribution), id: 0x228038df740

In [10]:
garch_arx_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_ARX_simulation)
garch_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_HARX_simulation)
garch_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_ARX_3_simulation)
garch_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_LS_simulation)
# garch_aim_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_AIM_simulation)



In [11]:
_garch_arx_vars = np.multiply(garch_arx_vars, -0.01)
_garch_harx_vars = np.multiply(garch_harx_vars, -0.01)
_garch_arx_3_vars = np.multiply(garch_arx_3_vars, -0.01)
_garch_ls_vars = np.multiply(garch_ls_vars, -0.01)
# _garch_aim_vars = np.multiply(garch_aim_vars, -0.01)

import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='GARCH(1, 1)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_arx_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='ARX-GARCH(1,1) (N)'
))


fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='HARX-GARCH(1,1) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='LS-GARCH(1,1) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5),
    name='ARX(3)-GARCH(1,1) (N)'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


    OTHER -ARCH models 
    

In [12]:
#EGARCH
EGARCH_ARX_simulation = model_construction('arx', 'EGARCH', 'norm', p=1, q=1, lags=[1, 5, 22])
EGARCH_HARX_simulation = model_construction('harx', 'EGARCH', 'norm', p=1, q=1, lags=[1, 5, 22])
EGARCH_ARX_3_simulation = model_construction('arx', 'EGARCH', 'norm', p=1, q=1, lags=3)
EGARCH_LS_simulation = model_construction('LS', 'EGARCH', 'norm', p=1, q=1, lags=None)
# EGARCH_AIM_simulation = model_construction('a-i-m', 'EGARCH', 'norm', p=1, q=1, lags=None)

EGARCHo1_HARX_simulation = model_construction('harx', 'EGARCH', 'norm', p=1, q=1, o=1, lags=[1, 5, 22])
EGARCHo1_ARX_3_simulation = model_construction('arx', 'EGARCH', 'norm', p=1, q=1, o=1, lags=3)
EGARCHo1_LS_simulation = model_construction('LS', 'EGARCH', 'norm', p=1, q=1, o=1, lags=None)

#EWMAVariance
EWMAVar_ARX_simulation = model_construction('arx', 'EWMAVariance', 'norm', lam=0.94, lags=[1, 5, 22])
EWMAVar_HARX_simulation = model_construction('harx', 'EWMAVariance', 'norm', lam=0.94, lags=[1, 5, 22])
EWMAVar_ARX_3_simulation = model_construction('arx', 'EWMAVariance', 'norm', lam=0.94, lags=3)
EWMAVar_LS_simulation = model_construction('LS', 'EWMAVariance', 'norm', lam=0.94, lags=None)
# EWMAVar_AIM_simulation = model_construction('a-i-m', 'EWMAVariance', 'norm', p=1, q=1, lags=None)

EWMAVar_MLE_HARX_simulation = model_construction('harx', 'EWMAVariance', 'norm', lam=None, lags=[1, 5, 22])
EWMAVar_MLE_ARX_3_simulation = model_construction('arx', 'EWMAVariance', 'norm', lam=None, lags=3)
EWMAVar_MLE_LS_simulation = model_construction('LS', 'EWMAVariance', 'norm', lam=None, lags=None)

#TARCH
TARCH_ARX_simulation = model_construction('arx', 'GARCH', 'norm', p=1, q=1, power=1.0, lags=[1, 5, 22])
TARCH_HARX_simulation = model_construction('harx', 'GARCH', 'norm', p=1, q=1, power=1.0, lags=[1, 5, 22])
TARCH_ARX_3_simulation = model_construction('arx', 'GARCH', 'norm', p=1, q=1, power=1.0, lags=3)
TARCH_LS_simulation = model_construction('LS', 'GARCH', 'norm', p=1, q=1, power=1.0, lags=None)
# TARCH_AIM_simulation = model_construction('a-i-m', 'GARCH', 'norm', p=1, q=1, power=1.0, lags=None)


In [13]:
#EGARCH
egarch_arx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_ARX_simulation)
_egarch_arx_vars = np.multiply(egarch_arx_vars, -0.01)

egarch_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_HARX_simulation)
_egarch_harx_vars = np.multiply(egarch_harx_vars, -0.01)

egarch_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_ARX_3_simulation)
_egarch_arx_3_vars = np.multiply(egarch_arx_3_vars, -0.01)

egarch_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_LS_simulation)
_egarch_ls_vars = np.multiply(egarch_ls_vars, -0.01)

# egarch_aim_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_AIM_simulation)
# _egarch_aim_vars = np.multiply(egarch_aim_vars, -0.01)


egarcho1_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCHo1_HARX_simulation)
_egarcho1_harx_vars = np.multiply(egarcho1_harx_vars, -0.01)

egarcho1_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCHo1_ARX_3_simulation)
_egarcho1_arx_3_vars = np.multiply(egarcho1_arx_3_vars, -0.01)

egarcho1_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCHo1_LS_simulation)
_egarcho1_ls_vars = np.multiply(egarcho1_ls_vars, -0.01)


#EWMAVariance
ewmavar_arx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_ARX_simulation)
_ewmavar_arx_vars = np.multiply(ewmavar_arx_vars, -0.01)

ewmavar_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_HARX_simulation)
_ewmavar_harx_vars = np.multiply(ewmavar_harx_vars, -0.01)

ewmavar_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_ARX_3_simulation)
_ewmavar_arx_3_vars = np.multiply(ewmavar_arx_3_vars, -0.01)

ewmavar_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_LS_simulation)
_ewmavar_ls_vars = np.multiply(ewmavar_ls_vars, -0.01)

# ewmavar_aim_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_AIM_simulation)
# _ewmavar_aim_vars = np.multiply(ewmavar_aim_vars, -0.01)


ewmavar_mle_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_MLE_HARX_simulation)
_ewmavar_mle_harx_vars = np.multiply(ewmavar_mle_harx_vars, -0.01)

ewmavar_mle_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_MLE_ARX_3_simulation)
_ewmavar_mle_arx_3_vars = np.multiply(ewmavar_mle_arx_3_vars, -0.01)

ewmavar_mle_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_MLE_LS_simulation)
_ewmavar_mle_ls_vars = np.multiply(ewmavar_mle_ls_vars, -0.01)


#TARCH
tarch_arx_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_ARX_simulation)
_tarch_arx_vars = np.multiply(tarch_arx_vars, -0.01)

tarch_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_HARX_simulation)
_tarch_harx_vars = np.multiply(tarch_harx_vars, -0.01)

tarch_arx_3_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_ARX_3_simulation)
_tarch_arx_3_vars = np.multiply(tarch_arx_3_vars, -0.01)

tarch_ls_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_LS_simulation)
_tarch_ls_vars = np.multiply(tarch_ls_vars, -0.01)

# tarch_aim_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_AIM_simulation)
# _tarch_aim_vars = np.multiply(tarch_aim_vars, -0.01)


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.


d:\Users\alexa\AppData\L

In [14]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='GARCH(1, 1)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_arx_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='ARX-EGARCH(1,0,1) (N)'
))


fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='HARX-EGARCH(1,0,1) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='LS-EGARCH(1,0,1) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5),
    name='ARX(3)-EGARCH(1,0,1) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarcho1_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5, dash='dot'),
    name='HARX-EGARCH(1,1,1) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarcho1_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='LS-EGARCH(1,1,1) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarcho1_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5, dash='dot'),
    name='ARX(3)-EGARCH(1,1,1) (N)'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [15]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='GARCH(1, 1)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_arx_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='ARX-EWMAVar(.94) (N)'
))


fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='HARX-EWMAVar(.94) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='LS-EWMAVar(.94) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5),
    name='ARX(3)-EWMAVar(.94) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_mle_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5, dash='dot'),
    name='HARX-EWMAVar(MLE) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_mle_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='LS-EWMAVar(MLE) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_mle_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5, dash='dot'),
    name='ARX(3)-EWMAVar(MLE) (N)'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [16]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='GARCH(1, 1)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_arx_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='ARX-TARCH(1,1) (N)'
))


fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='HARX-TARCH(1,1) (N))'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='LS-TARCH(1,1) (N)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_arx_3_vars, prices_usd),
    mode='lines',
    line=dict(color='brown', width=0.5),
    name='ARX(3)-TARCH(1,1) (N)'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [17]:
from GARCH_VaR_delete_or_merge import EGARCH_simulation

def EWMAVariance_type_simulation(lambda_):
    def simulate(returns, horizon=10, n_paths=50_000, alpha=0.01):
        vol = EWMAVariance(lambda_)
        model = ConstantMean(returns)
        model.volatility = vol
        model.distribution = Normal()
        results = model.fit(disp='off')
        h = int(horizon)                          
        forecast = results.forecast(horizon=h, method='simulation')          
        mu = results.params.get("mu", 0)
        sigma2 = forecast.variance.values[0]
        sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
        # nu = results.params.get('nu', 0)
        parameters = SimParams(volatility=sigma, mean=mu)

        sim_returns = solve_local_vol_gbm_log(parameters, 10, n_paths=n_paths)

        VaR = historical_var(sim_returns, alpha)
        
        return VaR

    return simulate

EWMAVariance94_simulation = EWMAVariance_type_simulation(0.94)
EWMAVarianceMLE_simulation = EWMAVariance_type_simulation(None)


def TARCH_simulation(returns, horizon=10, n_paths=50_000, granularity=1, alpha=0.01): 
    model = arch_model(returns, vol='GARCH', p=1, power=1.0, q=1, dist='normal')
    results = model.fit(disp='off')
    forecast = results.forecast(horizon=horizon, method='simulation')
    mu = results.params.get("mu", 0)
    sigma2 = forecast.variance.values[0]
    sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
    # nu = results.params.get('nu', 0)
    parameters = SimParams(volatility=sigma, mean=mu)

    # _ = solve_local_vol_gbm(parameters, 10, n_paths=n_paths)
    sim_returns = solve_local_vol_gbm_log(parameters, 10, n_paths=n_paths)

    # plt.hist(sim_returns, bins=50)
    # plt.show()

    VaR = historical_var(sim_returns, alpha)
    
    return VaR



In [18]:
ewma94_vars = fit_GARCH_VaR(log_returns_usd, EWMAVariance94_simulation)
ewmaMLE_vars = fit_GARCH_VaR(log_returns_usd, EWMAVarianceMLE_simulation)
tarch_vars = fit_GARCH_VaR(log_returns_usd, TARCH_simulation)
egarch_vars = fit_GARCH_VaR(log_returns_usd, EGARCH_simulation)

_ewma94_vars  = np.multiply(ewma94_vars, -1)
_ewmaMLE_vars = np.multiply(ewmaMLE_vars, -1)
_egarch_vars  = np.multiply(egarch_vars, -1)
_tarch_vars   = np.multiply(tarch_vars, -1)

d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:309: DataScaleWarning:

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 7.508e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:309: DataScaleWarning:

y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 7.505e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.


d:\Users\alexa\AppData\Local\Programs\Python\P

In [19]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewma94_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5),
    name='EWMAVar (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='EWMAVar'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='EGARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5, dash='dot'),
    name='EGARCH'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='TARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='TARCH'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [20]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewma94_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5),
    name='EWMAVar (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='EWMAVar'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='EGARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5, dash='dot'),
    name='EGARCH'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='TARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_ls_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='TARCH'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


    IN THE END AR models tend to understimate

    ARCH-in-model

In [33]:
#EGARCH
EGARCH_T_HARX_simulation = model_construction('harx', 'EGARCH', 't', p=1, q=1, lags=[1, 5, 22])

EWMAVar_T_HARX_simulation = model_construction('harx', 'EWMAVariance', 't', lam=0.94, lags=[1, 5, 22])

#TARCH
TARCH_T_HARX_simulation = model_construction('harx', 'GARCH', 't', p=1, q=1, power=1.0, lags=[1, 5, 22])

#EGARCH
EGARCH_T_simulation = model_construction('CM', 'EGARCH', 't', p=1, q=1, lags=[1, 5, 22])

EWMAVar_T_simulation = model_construction('CM', 'EWMAVariance', 't', lam=0.94, lags=[1, 5, 22])

#TARCH
TARCH_T_simulation = model_construction('CM', 'GARCH', 't', p=1, q=1, power=1.0, lags=[1, 5, 22])



In [34]:
#EGARCH
egarch_t_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_T_HARX_simulation)
_egarch_t_harx_vars = np.multiply(egarch_t_harx_vars, -0.01)

ewmavar_t_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_T_HARX_simulation)
_ewmavar_t_harx_vars = np.multiply(ewmavar_t_harx_vars, -0.01)

#TARCH
tarch_t_harx_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_T_HARX_simulation)
_tarch_t_harx_vars = np.multiply(tarch_t_harx_vars, -0.01)

#EGARCH
egarch_t_vars = fit_GARCH_VaR(log_returns_usd_scaled, EGARCH_T_simulation)
_egarch_t_vars = np.multiply(egarch_t_vars, -0.01)

ewmavar_t_vars = fit_GARCH_VaR(log_returns_usd_scaled, EWMAVar_T_simulation)
_ewmavar_t_vars = np.multiply(ewmavar_t_vars, -0.01)

#TARCH
tarch_t_vars = fit_GARCH_VaR(log_returns_usd_scaled, TARCH_T_simulation)
_tarch_t_vars = np.multiply(tarch_t_vars, -0.01)


d:\Users\alexa\AppData\Local\Programs\Python\Python312\Lib\site-packages\arch\univariate\base.py:768: ConvergenceWarning:

The optimizer returned code 9. The message is:
Iteration limit reached
See scipy.optimize.fmin_slsqp for code meaning.




In [35]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='GARCH(1,1)'
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_t_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5),
    name='EWMAVar (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_t_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='EWMAVar'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_t_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='EGARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_egarch_t_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5, dash='dot'),
    name='EGARCH'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_t_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='TARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_t_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='TARCH'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [36]:
from GARCH_VaR_delete_or_merge import student_GARCH_simulation

student_vars = fit_GARCH_VaR(log_returns_usd_scaled, student_GARCH_simulation)
_student_vars = np.multiply(student_vars, -0.01)

In [39]:
# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_student_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='GARCH(1,1) (S)'
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_t_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5),
    name='EWMAVar (Constant Mean)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_t_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dot'),
    name='EWMAVar'
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_ewmavar_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5, dash='dash'),
    name='EWMAVar (Normal)'
))

# fig.add_trace(go.Scatter(
#     x=np.arange(len(prices_usd)),
#     y=turn_var_into_real_terms(_egarch_t_vars, prices_usd),
#     mode='lines',
#     line=dict(color='blue', width=0.5),
#     name='EGARCH (Standard)'
# ))

# fig.add_trace(go.Scatter(
#     x=np.arange(len(prices_usd)),
#     y=turn_var_into_real_terms(_egarch_t_harx_vars, prices_usd),
#     mode='lines',
#     line=dict(color='blue', width=0.5, dash='dot'),
#     name='EGARCH'
# ))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_t_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5),
    name='TARCH (Standard)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_tarch_t_harx_vars, prices_usd),
    mode='lines',
    line=dict(color='green', width=0.5, dash='dot'),
    name='TARCH'
))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs with mean something',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


In [ ]:
def model_construction_debug(mean_process: str, volatiltiy_process: str, dist_process: str, lags=None, **kwargs): 
    def simulate(returns, horizon=10, n_paths=50_000, granularity=1, alpha=0.01): 
        vol = __VOLATILITY__[volatiltiy_process](**kwargs)
        if mean_process == 'CM' or mean_process == 'LS':
            model = __MEAN__[mean_process](returns)
        if mean_process == 'harx' or mean_process == 'a-i-m' or mean_process == 'arx':
            model = __MEAN__[mean_process](returns, lags)
        model.volatility = vol 
        model.distribution = __DISTRIBUTIONS__[dist_process]()
        results = model.fit(disp='off')
        h = int(horizon)                          
        forecast = results.forecast(horizon=h, method='simulation')          
        mu = results.params.get("mu", 0)
        sigma2 = forecast.variance.values[0]
        sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
        parameters = SimParams(volatility=sigma, mean=mu)

        if dist_process == 'norm': 
            return parameters
        
        if dist_process == 't': 
            nu = results.params.get('nu', 0)
            simulator = make_random_path_simulator_local_vol_student(parameters, 10, nu, granularity=granularity)
            sim_returns = _terminal_returns(simulator=simulator, n_paths=n_paths) 

            return parameters, nu



    return simulate

EWMAVar_normal_sim = model_construction_debug('CM', 'EWMAVariance', 'norm', lam=0.94)
# EWMAvar_t_sim = model_construction_debug('CM', 'EWMAVariance', 't', lam=0.94)
EWMAvar_t_sim = model_construction_debug('harx', 'EWMAVariance', 't', lam=0.94)


bench = model_construction_debug('CM', 'GARCH', 't', p=1, q=1)


In [ ]:
start=1000 
slice = log_returns_usd_scaled[start:start+365]

print(EWMAVar_normal_sim(slice))
print(EWMAvar_t_sim(slice))
print(bench(slice))

SimParams(volatility=array([1.03924564, 1.04101158, 1.04086786, 1.03938431, 1.03899595,
       1.04088305, 1.04038662, 1.03985252, 1.04289886, 1.04404954]), mean=np.float64(0.0297579255456663))
(SimParams(volatility=array([1.04264296, 1.04331172, 1.04132529, 1.04568951, 1.04365146,
       1.04368521, 1.04127477, 1.04162439, 1.04041025, 1.03858859]), mean=0), np.float64(4.638014011752184))
(SimParams(volatility=array([0.94240001, 0.94271634, 0.95125195, 0.92570837, 0.93060545,
       1.05238174, 1.04596745, 0.95612938, 0.93971297, 0.97620702]), mean=np.float64(0.05879959731736223)), np.float64(4.534648755378569))
